In [1]:
#chuyến kiểm thử id1459588

from pathlib import Path
import joblib
import numpy as np
import pandas as pd

candidates = [Path.cwd(), Path.cwd().parent,
              Path(r'C:\Users\Admin\Documents\TTNT\source\BTL\eta_predict')]
root = next(p for p in candidates if (p / 'models/random_forest.joblib').is_file())
model = joblib.load(root / 'models/random_forest.joblib')
for chunk in pd.read_csv(root / 'data/processed/test.csv', chunksize=25000):
    trip = chunk.loc[chunk['id'] == 'id1459588']
    if not trip.empty:
        break
else:
    raise ValueError('Không tìm thấy chuyến id1459588')
X = trip.loc[:, model.feature_names_in_]
print(X.to_string(index=False))

 haversine_km  manhattan_km  hour  dayofweek  hour_sin  hour_cos  is_rush_hour
     5.517153      5.991499     1          1  0.258819  0.965926             0


In [2]:
tree = model.estimators_[0].tree_
feature = X.columns[tree.feature[0]]
value = X[feature].item()
threshold = tree.threshold[0]
print('Đặc trưng:', feature)
print(f'Giá trị chuyến: {value:.4f} km. Ngưỡng đã học: {threshold:.4f} km.')
print('Con trái của nút 0:', tree.children_left[0])                                     #left
print('Con phải của nút 0:', tree.children_right[0])                                    #right
next_node = tree.children_left[0] if value <= threshold else tree.children_right[0]
print('Chuyến đi tới nút:', next_node)

Đặc trưng: haversine_km
Giá trị chuyến: 5.5172 km. Ngưỡng đã học: 4.4729 km.
Con trái của nút 0: 1
Con phải của nút 0: 185966
Chuyến đi tới nút: 185966


In [3]:
pending = [tree.children_left[0]]
left_count = 0
while pending:
    node = pending.pop()
    left_count += 1
    if tree.children_left[node] != tree.children_right[node]:
        pending.extend([tree.children_left[node], tree.children_right[node]])
print('Số nút trong cây con bên trái:', left_count)
print('Mã nút con bên phải của gốc:', tree.children_right[0])

Số nút trong cây con bên trái: 185965
Mã nút con bên phải của gốc: 185966


In [4]:
#quan trong: Mỗi dòng ghi một phép so sánh
node = 0
while tree.children_left[node] != tree.children_right[node]:
    name = X.columns[tree.feature[node]]
    value = np.float32(X[name].item()).item()
    threshold = tree.threshold[node]
    left = value <= threshold
    child = tree.children_left[node] if left else tree.children_right[node]
    sign = '<=' if left else '>'
    print(f'Nút {node}: {name} = {value:.5f} {sign} {threshold:.5f} → nút {child}')
    node = child
print(f'Lá {node}: dự đoán của cây 1 là {tree.value[node].item():.1f} giây')

Nút 0: haversine_km = 5.51715 > 4.47291 → nút 185966
Nút 185966: haversine_km = 5.51715 <= 13.31129 → nút 185967
Nút 185967: haversine_km = 5.51715 <= 8.20421 → nút 185968
Nút 185968: hour = 1.00000 <= 7.50000 → nút 185969
Nút 185969: haversine_km = 5.51715 <= 6.22322 → nút 185970
Nút 185970: haversine_km = 5.51715 > 5.29501 → nút 188074
Nút 188074: hour_cos = 0.96593 > -0.12941 → nút 188348
Nút 188348: hour_cos = 0.96593 > 0.91598 → nút 189228
Nút 189228: dayofweek = 1.00000 <= 2.50000 → nút 189229
Nút 189229: manhattan_km = 5.99150 <= 7.13357 → nút 189230
Nút 189230: dayofweek = 1.00000 <= 1.50000 → nút 189231
Nút 189231: haversine_km = 5.51715 <= 5.61285 → nút 189232
Nút 189232: haversine_km = 5.51715 <= 5.58811 → nút 189233
Nút 189233: haversine_km = 5.51715 > 5.39752 → nút 189241
Nút 189241: manhattan_km = 5.99150 <= 6.03687 → nút 189242
Lá 189242: dự đoán của cây 1 là 882.6 giây


In [5]:
#forest
values = X.to_numpy()
per_tree = np.array([t.predict(values)[0] for t in model.estimators_])
prediction = model.predict(X)[0]
print('Số cây:', len(per_tree))
print('Ba cây đầu (giây):', per_tree[:3].round(3))
print(f'Trung bình các cây: {per_tree.mean():.9f} giây')
print(f'model.predict(): {prediction:.9f} giây')
print('Thời lượng hiển thị:', round(prediction), 'giây')
print('Thời lượng thực tế:', trip['trip_duration'].item(), 'giây')
np.testing.assert_allclose(per_tree.mean(), prediction, rtol=0, atol=1e-8)

Số cây: 100
Ba cây đầu (giây): [ 882.6   1131.2   1000.889]
Trung bình các cây: 1013.684500126 giây
model.predict(): 1013.684500126 giây
Thời lượng hiển thị: 1014 giây
Thời lượng thực tế: 778 giây
